In [0]:
pip install great_expectations


In [0]:
%restart_python

In [0]:
import os
import pandas as pd
from pyspark.sql import SparkSession

# Directorio base y rutas de archivo
BASE_DIR = "/Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas"
SQL_FILE = os.path.join(BASE_DIR, "src/queries/query.sql")
CARPETA_SALIDA = os.path.join(BASE_DIR, "data/raw")
os.makedirs(CARPETA_SALIDA, exist_ok=True)

# Crear sesión de Spark
spark = SparkSession.builder.getOrCreate()

# ╭───────────── 1) Leer la consulta SQL ─────────────╮
with open(SQL_FILE, "r", encoding="utf-8") as f:
    consulta_sql = f.read()

# ╭───────────── 2) Ejecutar la consulta en Spark ─────╮
df_spark = spark.sql(consulta_sql)

# ╭───────────── 3) Convertir el resultado a Pandas ───╮
df = df_spark.toPandas()

# ╭───────────── 4) Guardar el DataFrame en formato Parquet ─╮
ruta_parquet = os.path.join(CARPETA_SALIDA, "usuarios_unicos.parquet")
df.to_parquet(ruta_parquet, index=False)
print(f"Archivo guardado en: {ruta_parquet}")

# ╭───────────── 5) Leer el archivo Parquet para verificar ─╮
df_verificacion = pd.read_parquet(ruta_parquet)
print(df_verificacion.head())


# Genera datos sintéticos


In [0]:
%pip install sdv

In [0]:
#!pip install sdv
%restart_python


In [0]:
df.shape

In [0]:
# ╭───────────── 1) Importar librerías ─────────────╮
import pandas as pd
from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

# ╭───────────── 2) Cargar los datos reales ─────────────╮
df = pd.read_parquet("/Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas/data/raw/usuarios_unicos.parquet")

# ╭──── 3) Inferir el metadata del DataFrame ────────────╮
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)

# ╭──── 4) Entrenar el modelo GaussianCopulaSynthesizer ─╮
synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(df)

# ╭───── 5) Generar 200 filas de datos sintéticos ────────╮
synthetic_df = synthesizer.sample(500)

# ╭────────── 6) Verificar el resultado ───────────────╮
print(synthetic_df.head())
print("Forma del DataFrame sintético:", synthetic_df.shape)
synthetic_df.shape
synthetic_df.to_parquet("/Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas/data/raw/usuarios_unicos_sinte.parquet")
df = synthetic_df.copy()
